In [0]:
%pip install lightgbm

In [0]:
import pandas as pd
import numpy as np
import lightgbm

import mlflow
import mlflow.lightgbm

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from lightgbm import LGBMRegressor

print("Libraries loaded successfully")

In [0]:
DATA_PATH = "demand_forecasting.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))




In [0]:
df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop=True)

print("Date conversion and sorting completed.")

In [0]:
GROUP_COLS = ["Store ID","Product ID"]

print("Groups:", GROUP_COLS)

In [0]:
df["day_of_week"] = df["Date"].dt.dayofweek

df["day_of_month"] = df["Date"].dt.day

df["week_of_year"] = (
    df["Date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

df["month"] = df["Date"].dt.month

df["quarter"] = df["Date"].dt.quarter

df["year"] = df["Date"].dt.year

df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)


In [0]:
df["lag_1"] = (df.groupby(GROUP_COLS)["Demand"].shift(1))

df["lag_7"] = (df.groupby(GROUP_COLS)["Demand"].shift(7))

df["lag_14"] = (df.groupby(GROUP_COLS)["Demand"].shift(14))

df["lag_28"] = (df.groupby(GROUP_COLS)["Demand"].shift(28))

In [0]:
df["rolling_mean_7"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(7)
           .mean()
      )
)

df["rolling_mean_14"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(14)
           .mean()
      )
)

df["rolling_mean_28"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(28)
           .mean()
      )
)


In [0]:
df["rolling_std_7"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(7)
           .std()
      )
)

df["rolling_std_14"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(14)
           .std()
      )
)

In [0]:
HISTORY_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14"
]

df_model = df.dropna(
    subset=HISTORY_FEATURES
).copy()

print("Original rows:", len(df))
print("Rows after feature engineering:", len(df_model))

In [0]:
LAG_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28"
]

ROLLING_FEATURES = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14"
]

BUSINESS_FEATURES = [
    "Inventory Level",
    "Units Ordered",
    "Price",
    "Discount",
    "Promotion",
    "Competitor Pricing"
]

TIME_FEATURES = [
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "quarter",
    "year",
    "is_weekend"
]

CATEGORICAL_FEATURES = [
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality",
    "Epidemic"
]

FEATURES = (
    LAG_FEATURES
    + ROLLING_FEATURES
    + BUSINESS_FEATURES
    + TIME_FEATURES
    + CATEGORICAL_FEATURES
)

TARGET = "Demand"

print("Number of features:", len(FEATURES))

print("\nFeatures:")
for feature in FEATURES:
    print("-", feature)

print("\nTarget:", TARGET)

In [0]:
missing_features = [
    feature
    for feature in FEATURES
    if feature not in df_model.columns
]

if missing_features:

    raise ValueError(
        f"Missing features: {missing_features}"
    )

else:

    print("All features are available.")

In [0]:
dates = sorted(df_model["Date"].unique())

train_end = dates[int(len(dates) * 0.70)]

valid_end = dates[int(len(dates) * 0.85)]

print("Train end:", train_end)
print("Validation end:", valid_end)

In [0]:
train_df = df_model[df_model["Date"] <= train_end].copy()

In [0]:
valid_df = df_model[(df_model["Date"] > train_end) & (df_model["Date"] <= valid_end)].copy()

In [0]:
test_df = df_model[df_model["Date"] > valid_end].copy()

In [0]:
print("DATA SPLIT")
print("================================")

print("Train      :", train_df.shape)
print("Validation :", valid_df.shape)
print("Test       :", test_df.shape)

print("================================")

In [0]:
X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()

X_valid = valid_df[FEATURES].copy()
y_valid = valid_df[TARGET].copy()

X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [0]:
for col in CATEGORICAL_FEATURES:

    X_train[col] = (X_train[col].astype("category"))

    X_valid[col] = (X_valid[col].astype("category"))

    X_test[col] = (X_test[col].astype("category"))

print("Categorical columns converted.")


In [0]:
EXPERIMENT_NAME = ("/Shared/demand-forecasting")

mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow experiment:",EXPERIMENT_NAME)


In [0]:
MODEL_PARAMS = {
    "objective": "regression",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

print("Model parameters:")

for key, value in MODEL_PARAMS.items():
    print(f"{key}: {value}")

In [0]:
from mlflow.models import infer_signature

In [0]:
with mlflow.start_run(run_name="lightgbm_demand_forecasting"):

    # Create model
    model = LGBMRegressor(**MODEL_PARAMS)

    # Train model
    model.fit(X_train,y_train,categorical_feature=CATEGORICAL_FEATURES)
    signature = infer_signature(X_train,model.predict(X_train))
    
    input_example = X_train.iloc[[0]]

    mlflow.lightgbm.log_model(
        model,
        name="model",
        signature=signature,
        input_example=input_example
        )

    print("Model trained successfully.")


    # --------------------------------------------------------
    # Validation predictions
    # --------------------------------------------------------

    valid_predictions = model.predict(X_valid)

    # Demand should not be negative
    valid_predictions = np.maximum(valid_predictions,0)


    # --------------------------------------------------------
    # Validation metrics
    # --------------------------------------------------------

    validation_mae = mean_absolute_error(y_valid,valid_predictions)

    validation_rmse = np.sqrt(mean_squared_error(y_valid,valid_predictions))

    validation_r2 = r2_score(y_valid,valid_predictions)


    # Safe MAPE
    mask = y_valid != 0

    validation_mape = (
        np.mean(
            np.abs(
                (
                    y_valid[mask]
                    -
                    valid_predictions[mask]
                )
                /
                y_valid[mask]
            )
        )
        * 100
    )


    # --------------------------------------------------------
    # Log parameters
    # --------------------------------------------------------

    mlflow.log_params(MODEL_PARAMS)

    mlflow.log_param("target",TARGET)

    mlflow.log_param("num_features",len(FEATURES))

    mlflow.log_param("split_type","time_based")


    # --------------------------------------------------------
    # Log metrics
    # --------------------------------------------------------

    mlflow.log_metric("validation_mae",validation_mae)

    mlflow.log_metric("validation_rmse",validation_rmse)

    mlflow.log_metric("validation_r2",validation_r2)

    mlflow.log_metric("validation_mape",validation_mape)


    # --------------------------------------------------------
    # Log model
    # --------------------------------------------------------

    mlflow.lightgbm.log_model(model,"model")


    # --------------------------------------------------------
    # Print results
    # --------------------------------------------------------

    print("")
    print("================================")
    print("VALIDATION RESULTS")
    print("================================")

    print("MAE  :",validation_mae)

    print("RMSE :",validation_rmse)

    print("R2   :",validation_r2)

    print("MAPE :",validation_mape)

    print("================================")

In [0]:
baseline_predictions = (valid_df["lag_1"].values)

baseline_predictions = np.maximum(baseline_predictions,0)

baseline_mae = mean_absolute_error(y_valid,baseline_predictions)

baseline_rmse = np.sqrt(mean_squared_error(y_valid,baseline_predictions))

print("================================")
print("BASELINE RESULTS")
print("================================")

print("Baseline MAE :",baseline_mae)

print("Baseline RMSE:",baseline_rmse)

print("================================")


In [0]:
print("================================")
print("MODEL COMPARISON")
print("================================")

print("Baseline MAE :",baseline_mae)

print("LightGBM MAE  :",validation_mae)

print("Baseline RMSE:",baseline_rmse)

print("LightGBM RMSE :",validation_rmse)

print("================================")


if validation_mae < baseline_mae:

    print("LightGBM improved over the baseline.")

else:

    print("LightGBM did NOT improve over the baseline.")

In [0]:
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_
})

importance = importance.sort_values("importance",ascending=False)

display(importance.head(20))

In [0]:
test_predictions = model.predict(X_test)

test_predictions = np.maximum(test_predictions,0)

test_mae = mean_absolute_error(y_test,test_predictions)

test_rmse = np.sqrt(mean_squared_error(y_test,test_predictions))

test_r2 = r2_score(y_test,test_predictions)

test_mask = y_test != 0

test_mape = (
    np.mean(
        np.abs(
            (
                y_test[test_mask]
                -
                test_predictions[test_mask]
            )
            /
            y_test[test_mask]
        )
    )
    * 100
)


print("================================")
print("FINAL TEST RESULTS")
print("================================")

print("Test MAE  :",test_mae)

print("Test RMSE :",test_rmse)

print("Test R2   :",test_r2)

print("Test MAPE :",test_mape)

print("================================")


In [0]:
print("MODEL TRAINING COMPLETED SUCCESSFULLY")
print("============================================")

print("Model      : LightGBM")
print("Experiment :", EXPERIMENT_NAME)

print("")
print("Validation MAE :", validation_mae)
print("Validation RMSE:", validation_rmse)

print("")
print("Test MAE       :", test_mae)
print("Test RMSE      :", test_rmse)

print("============================================")